In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
biomass_l1c_access.py
=====================
Discover, retrieve and pre-build ESA BIOMASS L1C (STA) tomographic stacks.

Turns a track/frame or AOI into a validated, coregistered quad-pol cube with
master-referenced vertical wavenumbers — the inputs a TomoSAR pipeline needs.

Quick start
-----------
Stream a window directly from the catalogue, no product download::

    from biomass_l1c_access import build_stack

    out = build_stack(
        track=29, frame=1, major_cycle=5,
        max_acquisitions=7, strategy="spread",
        az_off=8000, az_size=2048,
        stream=True, auth=False,
        save_netcdf="amacayacu_m05_stack.nc",
    )
    ds = out["dataset"]          # xarray: HH HV VH VV kz mask P1 P2 XP

Reload later without touching the network::

    from biomass_l1c_access import load_dataset
    ds = load_dataset("amacayacu_m05_stack.nc")
    ds = out["stack"].add_pauli(ds)      # P1/P2/XP are not stored on disk

Discovery routes, most to least reliable
----------------------------------------
``track=`` / ``frame=``
    One CQL2 attribute query. No pagination, no spatial index. Preferred.
``reference_id=``
    Parses geometry from a known product ID. Fails if that product was
    superseded by a newer processing baseline.
``aoi=``
    Samples a bounded window and intersects footprints client-side.
    Server-side bbox filtering returns zero matches on this catalogue, and
    unbounded paging through ~182,000 items triggers HTTP 502. Resolve once,
    then pass track/frame directly.

Retrieval modes
---------------
``stream=True``
    Windowed HTTPS range reads of the ``_abs``/``_phase`` rasters. Transfers
    only the pixels requested — ~0.6 GB for a 7 x 2048 x 1369 window versus
    ~56 GB of whole products. Parallelised across acquisitions.
``stream=False``
    Downloads and unpacks full products. Needed for the DEM, annotation XML
    and other assets the streaming path skips.

Domain notes
------------
Stack
    A set of repeat cycles (C01, C02, ...) sharing global coverage, major
    cycle, track, frame and processing baseline. These are mutually
    coregistered within an L1C STA product and need no further resampling.

Minimum acquisitions
    Three. Two give a single interferogram — a phase centre, not a vertical
    profile. BIOMASS interferometric phase is nominally 3 cycles, tomographic
    phase nominally 7.

Polarisation
    Lives in the *band* dimension of the measurement raster
    (``PolarisationsSequence = 'HH HV VH VV'``), not in the filename. Band
    order is verified by physics in :meth:`BiomassL1CStack.summary`:
    reciprocity ``HV - VH`` near 0 dB, canopy depolarisation ``HV/HH``
    between -14 and -3 dB, and positive ``HH - VV`` from the trunk-ground
    double bounce.

Sentinel
    ``9999.0`` marks invalid pixels at swath edges. The validity mask
    requires a pixel to be finite, non-zero and sub-sentinel in *every*
    acquisition and polarisation.

Vertical wavenumbers
    ``waveNumbers`` LUTs are subsampled relative to the measurement raster,
    so they must be interpolated with the window offset applied — otherwise
    the whole scene is stretched across the subset, injecting a spurious
    ramp. Values are referenced to a master acquisition, chosen by default as
    the one closest to zero, which minimises the maximum baseline.

    Vertical resolution follows from the total spread,
    :math:`\\delta z = 2\\pi / \\Delta k_z`; the height ambiguity follows
    from the smallest gap between adjacent wavenumbers.

Pauli channels
    ``P2 = (HH - VV)/sqrt(2)`` captures the trunk-ground double bounce and
    locates the **ground**; ``XP = (HV + VH)/sqrt(2)`` captures volume
    scattering and locates the **canopy**. Canopy height follows from the
    peak separation, which is insensitive to a common phase offset such as a
    residual flattening screen.

Known catalogue quirks
----------------------
* Server-side ``bbox`` returns zero matches — filter client-side.
* Sustained pagination returns HTTP 502; ``aoi_to_tracks`` samples a bounded
  window and tolerates mid-stream failure.
* The ``/data/zipper/`` endpoint assembles archives on demand and
  intermittently returns HTTP 500. ``_download_fallback`` retries with
  exponential backoff.
* GDAL cannot resolve a VRT's sibling TIFFs through a
  ``/vsipythonfilelike/`` stream, so the streaming path reads the
  ``_abs``/``_phase`` pair directly and recombines as :math:`A e^{j\\phi}`.
* Reprocessed duplicates persist in the catalogue; ``dedupe_most_recent``
  keeps the newest by decoding the base-36 processing timestamp.

Requirements
------------
Core
    numpy, requests, pystac-client, shapely, scipy
Raster I/O
    rasterio, fsspec (streaming); GDAL/osgeo (local reads)
Optional
    xarray + h5netcdf or netCDF4 for Dataset export; h5py for LUT reads;
    tqdm for progress bars; ``BPS_inputs_download`` if available, else the
    built-in parallel downloader is used.

Authentication
--------------
Expects ``$MAAP_CATALOGUE_ACCESS_TOKEN``. Either set it externally and pass
``auth=False``, or let :class:`MaapAuth` mint one from an offline token plus
``$MAAP_CLIENT_ID`` / ``$MAAP_CLIENT_SECRET``. Access tokens are short-lived
— refresh before long streaming or download runs.

References
----------
ESA BIOMASS Level-1c Stack Product ATBD — coregistration and ``waveNumbers``
definitions. Level-1c processing uses the Copernicus DEM for ground steering.

Author
------
<your name> <your.email@institution>

Created
-------
2026-10-16
"""
from __future__ import annotations


import os
import re
import glob
import pathlib
import logging
import itertools
import time
import threading
import inspect
import importlib
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Sequence
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import distance_transform_edt
import subprocess
from osgeo import gdal
from tqdm import tqdm
import numpy as np
import requests
from pystac_client import Client
from shapely.geometry import shape, box
import fsspec
import rasterio as rio
from rasterio.windows import Window
import xarray as xr
import h5py




log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for _noisy in ("rasterio._filepath", "rasterio._env", "rasterio._io",
               "rasterio.session", "fsspec"):
    logging.getLogger(_noisy).setLevel(logging.CRITICAL)
# Config
CATALOG_URL = "https://catalog.maap.eo.esa.int/catalogue/"
IAM_URL = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"

L1C_COLLECTIONS = ["BiomassLevel1c", "BiomassLevel1cIOC"]
POLS = ("HH", "HV", "VH", "VV")          # == PolarisationsSequence
SENTINEL = 9999.0
MIN_CYCLES = 3

# BIO_S1_STA__1S_<start>_<stop>_<T|I>_G01_M02_C01_T029_F001_01_<hash>
_STA_RE = re.compile(
    r"BIO_(?P<swath>S\d)_STA__1S_"
    r"(?P<start>\d{8}T\d{6})_(?P<stop>\d{8}T\d{6})_"
    r"(?P<phase>[TI])_G(?P<glob>\d{2})_M(?P<major>\d{2})_C(?P<cycle>\d{2})_"
    r"T(?P<track>\d{3})_F(?P<frame>\d{3})_(?P<baseline>\d{2})_"
)


"""Mirror get_stack(): resolve one ID, then CQL2-match G/M/T/F/baseline."""
_SCS_ID_PATTERN = re.compile(
    r'_(?P<phase>[TI])_G(?P<glob>\d{2})_M(?P<major>\d{2})_C(?P<cycle>\d{2})'
    r'_T(?P<track>\d{3})_F(?P<frame>[\d_]{3,4})_(?P<baseline>\d{2})_')


# Server Authentication
class MaapAuth:
    """Exchange a long-lived offline token for short-lived access tokens.

    The ESA MAAP IAM endpoint issues access tokens valid for minutes to an
    hour. This class holds the offline (refresh) token and the OAuth client
    credentials, and mints fresh access tokens on demand.

    Parameters
    ----------
    token_file : str, default 'token.txt'
        Path to the offline token. Searched in the working directory, then
        ``/home/jovyan``.
    client_id : str, optional
        OAuth client ID. Falls back to ``$MAAP_CLIENT_ID``.
    client_secret : str, optional
        OAuth client secret. Falls back to ``$MAAP_CLIENT_SECRET``.

    Raises
    ------
    FileNotFoundError
        If no offline token file is found.
    ValueError
        If the offline token, client ID or client secret is missing.

    Notes
    -----
    If ``$MAAP_CATALOGUE_ACCESS_TOKEN`` is already set by another part of
    the session, ``retrieve(auth=False)`` reuses it and this class is not
    needed at all.
    """

    def __init__(self, token_file="token.txt", client_id=None, client_secret=None):
        self.client_id = client_id or os.environ.get("MAAP_CLIENT_ID")
        self.client_secret = client_secret or os.environ.get("MAAP_CLIENT_SECRET")
        p = pathlib.Path(token_file)
        if not p.exists():
            p = pathlib.Path("/home/jovyan") / token_file
        if not p.exists():
            raise FileNotFoundError(f"No token file: {token_file} (cwd={os.getcwd()})")
        self.offline_token = p.read_text().strip().replace("\n", "")
        if not all([self.offline_token, self.client_id, self.client_secret]):
            raise ValueError("Missing OFFLINE_TOKEN, CLIENT_ID or CLIENT_SECRET")

    def token(self) -> str:
        """Mint a fresh access token and publish it to the environment.

        Returns
        -------
        str
            The access token. Also written to
            ``$MAAP_CATALOGUE_ACCESS_TOKEN`` so that downloaders and
            streaming readers pick it up without being passed explicitly.

        Raises
        ------
        requests.HTTPError
            If the IAM endpoint rejects the refresh request.
        RuntimeError
            If the response contains no ``access_token`` field.
        """
        
        r = requests.post(IAM_URL, data={
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            "grant_type": "refresh_token",
            "refresh_token": self.offline_token,
            "scope": "offline_access openid",
        })
        r.raise_for_status()
        tok = r.json().get("access_token")
        if not tok:
            raise RuntimeError("No access_token in IAM response")
        os.environ["MAAP_CATALOGUE_ACCESS_TOKEN"] = tok
        return tok


# Metadata
@dataclass(frozen=True)
class StaProduct:
    """One repeat-cycle acquisition within a coregistered L1C stack.

    Fields are parsed from the product ID, which encodes the full
    acquisition geometry.

    Attributes
    ----------
    item_id : str
        Full STAC item ID, e.g. ``BIO_S1_STA__1S_..._C01_T029_F001_02_DZ1IYD``.
    href : str
        URL of the product ``.zip`` asset.
    swath : str
        Swath identifier, e.g. ``'S1'``.
    phase : str
        ``'T'`` for tomographic phase, ``'I'`` for interferometric.
    glob, major : str
        Global coverage and major cycle identifiers, zero-padded.
    cycle : int
        Repeat cycle number within the major cycle (1-based).
    track, frame : str
        Orbit track and frame, zero-padded to 3 digits.
    baseline : str
        Processing baseline / version, e.g. ``'02'``.
    t_start : datetime
        Acquisition start time.
    local_dir : str or None
        Unpacked product directory once downloaded.
    """
    
    item_id: str
    href: str
    swath: str
    phase: st
    glob: str
    major: str
    cycle: int
    track: str
    frame: str
    baseline: str
    t_start: datetime
    local_dir: str | None = None

    @classmethod
    def from_item(cls, item) -> "StaProduct":
        """Parse a STAC item into a StaProduct.

        Parameters
        ----------
        item : pystac.Item
            Item from a catalogue search.

        Returns
        -------
        StaProduct

        Raises
        ------
        ValueError
            If the item ID does not match the expected STA naming pattern.
        """
        m = _STA_RE.search(item.id)
        if not m:
            raise ValueError(f"Unparseable STA id: {item.id}")
        g = m.groupdict()
        return cls(
            item_id=item.id,
            href=item.assets["product"].href,
            swath=g["swath"], phase=g["phase"], glob=g["glob"], major=g["major"],
            cycle=int(g["cycle"]), track=g["track"], frame=g["frame"],
            baseline=g["baseline"],
            t_start=datetime.strptime(g["start"], "%Y%m%dT%H%M%S"),
        )

    @property
    def stack_key(self) -> tuple:
        """Identify the coregistered stack this product belongs to.

        Returns
        -------
        tuple
            ``(swath, phase, glob, major, track, frame, baseline)``.
            Products sharing this key are mutually coregistered and can be
            stacked without further resampling.
        """
        return (self.swath, self.phase, self.glob, self.major,
                self.track, self.frame, self.baseline)


def _decode_ts(item_id: str) -> datetime:
    """Decode the base-36 processing timestamp from a product ID.

    Parameters
    ----------
    item_id : str
        Product ID whose final underscore-delimited field is a base-36
        offset in seconds from 2000-01-01.

    Returns
    -------
    datetime
        Processing time, used to rank reprocessed duplicates.
    """
    return datetime(2000, 1, 1) + timedelta(seconds=int(item_id.split("_")[-1], 36))


def dedupe_most_recent(items):
    """Collapse reprocessed duplicates, keeping the newest of each.

    The catalogue retains superseded reprocessings. Items differing only in
    their base-36 suffix describe the same acquisition.

    Parameters
    ----------
    items : list of pystac.Item

    Returns
    -------
    list of pystac.Item
        One item per acquisition, the most recently processed.
    """
    best = {}
    for it in items:
        key = it.id.rsplit("_", 1)[0]
        ts = _decode_ts(it.id)
        if key not in best or ts > best[key][1]:
            best[key] = (it, ts)
    return [i for i, _ in best.values()]


# Discovery
MAX_SAFE_ITEMS = 5000


def find_stacks(bbox=None, datetime_range=None, swath=None, phase=None,
                min_cycles=MIN_CYCLES, catalog_url=CATALOG_URL,
                max_items=None, retries=3):
    """Search the catalogue and group products into coregistered stacks.

    Spatial filtering is performed **client-side**: server-side ``bbox``
    returns zero matches on this catalogue.

    Parameters
    ----------
    bbox : sequence of float, optional
        ``[W, S, E, N]`` in degrees. Applied after retrieval by intersecting
        item footprints locally.
    datetime_range : sequence of str, optional
        ``[start, end]`` ISO-8601. Strongly recommended — see Notes.
    swath : str, optional
        Keep only this swath, e.g. ``'S1'``.
    phase : {'T', 'I'}, optional
        Keep only tomographic or interferometric acquisitions.
    min_cycles : int, default 3
        Discard stacks with fewer repeat cycles.
    catalog_url : str
        STAC endpoint.
    max_items : int, optional
        Hard cap on items paged. Required if no ``datetime_range`` is given.
    retries : int, default 3
        Retry attempts with exponential backoff on transient paging failures.

    Returns
    -------
    list of list of StaProduct
        Stacks sorted by size, descending. Each inner list is cycle-sorted
        and deduplicated.

    Raises
    ------
    ValueError
        If the query is unbounded, or if more than ``MAX_SAFE_ITEMS`` match
        without an explicit ``max_items``.

    Notes
    -----
    An unbounded query pages through roughly 182,000 items and reliably
    triggers HTTP 502 from the catalogue gateway. Prefer
    :func:`find_stacks_by_gmtf` when the track and frame are known.
    """
    
    if datetime_range is None and bbox is None and max_items is None:
        raise ValueError(
            "Unbounded query would page through the entire L1C archive "
            "(~182,000 items) and trigger a 502. Supply datetime_range, "
            "or set max_items explicitly.")

    catalog = Client.open(catalog_url)
    kw = dict(collections=L1C_COLLECTIONS, method="GET")
    if datetime_range:
        kw["datetime"] = list(datetime_range)
    if max_items:
        kw["max_items"] = max_items

    search = catalog.search(**kw)
    n_matched = search.matched()
    log.info("Catalogue matched %d items (time filter only)", n_matched)

    if n_matched > MAX_SAFE_ITEMS and not max_items:
        raise ValueError(
            f"{n_matched} items matched — too many to page reliably "
            f"(limit {MAX_SAFE_ITEMS}). Narrow datetime_range, or pass "
            f"max_items=N, or use from_reference_id() instead.")

    # paginating including retry and backoff logic
    raw = None
    for attempt in range(retries):
        try:
            raw = list(search.items())
            break
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt
            log.warning("Paging failed (%s) — retry %d/%d in %ds",
                        type(e).__name__, attempt + 1, retries, wait)
            time.sleep(wait)
            search = catalog.search(**kw)

    items = dedupe_most_recent(raw)
    log.info("After dedupe: %d", len(items))

    # client-side spatial intersection
    if bbox is not None:
        aoi = box(*bbox)
        keep = []
        for it in items:
            if it.geometry is None:
                continue
            try:
                if shape(it.geometry).intersects(aoi):
                    keep.append(it)
            except Exception as e:
                log.debug("geometry fail %s: %s", it.id, e)
        log.info("Spatial filter: %d → %d items", len(items), len(keep))
        items = keep

    groups = {}
    for it in items:
        try:
            p = StaProduct.from_item(it)
        except (ValueError, KeyError) as e:
            log.warning("unparsed %s: %s", it.id, e)   # was debug — now visible
            continue
        if swath and p.swath != swath:
            continue
        if phase and p.phase != phase:
            continue
        groups.setdefault(p.stack_key, []).append(p)

    stacks = []
    for key, prods in groups.items():
        prods = sorted({p.cycle: p for p in prods}.values(), key=lambda x: x.cycle)
        if len(prods) >= min_cycles:
            stacks.append(prods)

    stacks.sort(key=len, reverse=True)
    log.info("%d stack(s) with >= %d cycles", len(stacks), min_cycles)
    for s in stacks[:10]:
        p = s[0]
        log.info("  %s G%s M%s T%s F%s | %s | %d cycles | %d d | %s",
                 p.swath, p.glob, p.major, p.track, p.frame, p.phase,
                 len(s), (s[-1].t_start - s[0].t_start).days, p.t_start.date())
    return stacks


def find_stack_by_id(reference_id, catalog_url=CATALOG_URL,
                     min_cycles=MIN_CYCLES, return_items=False):
    """Resolve a complete stack from a single reference product ID.

    Parses the geometry out of the reference ID, then issues one CQL2
    attribute query for all products sharing that geometry and baseline.

    Parameters
    ----------
    reference_id : str
        Any product ID belonging to the desired stack.
    catalog_url : str
        STAC endpoint.
    min_cycles : int, default 3
        Minimum repeat cycles required.
    return_items : bool, default False
        Also return the underlying STAC items, needed for streaming.

    Returns
    -------
    list of StaProduct
        Cycle-sorted products, or ``(products, items)`` when
        ``return_items=True``.

    Raises
    ------
    IndexError
        If no item matches ``reference_id``. Common cause: the product was
        superseded by a newer processing baseline.
    RuntimeError
        If fewer than ``min_cycles`` cycles are available.
    """
    
    catalog = Client.open(catalog_url)
    hits = list(catalog.search(collections=L1C_COLLECTIONS,
                               ids=reference_id, method="GET").items())
    if not hits:
        raise IndexError(f"No L1C item with id {reference_id}")

    prod_type = hits[0].properties.get("product:type")
    g = _SCS_ID_PATTERN.search(reference_id).groupdict()
    log.info("Reference: type=%s G%s M%s T%s F%s baseline=%s", prod_type,
             g["glob"], g["major"], g["track"], g["frame"], g["baseline"])

    search = catalog.search(
        collections=L1C_COLLECTIONS,
        filter=(f"product:type='{prod_type}' and "
                f"eofeos:global_coverage_id={int(g['glob'])} and "
                f"eofeos:major_cycle_id={int(g['major'])} and "
                f"track={int(g['track'])} and frame={int(g['frame'])} and "
                f"version='{g['baseline']}'"),
        method="GET")

    items = dedupe_most_recent(list(search.items()))
    by_id = {it.id: it for it in items}
    prods = sorted({p.cycle: p for p in
                    (StaProduct.from_item(it) for it in items)}.values(),
                   key=lambda x: x.cycle)
    if len(prods) < min_cycles:
        raise RuntimeError(f"Only {len(prods)} cycles (need >= {min_cycles})")
    log.info("Stack: C%s | span %d days",
             "/".join(f"{p.cycle:02d}" for p in prods),
             (prods[-1].t_start - prods[0].t_start).days)
    return (prods, [by_id[p.item_id] for p in prods]) if return_items else prods


def select_baseline(stack: list[StaProduct],
                    cycles: Sequence[int] | None = None,
                    max_acquisitions: int | None = None,
                    max_temporal_baseline: float | None = None,
                    strategy: str = "shortest") -> list[StaProduct]:
    """Select a sub-baseline from a full stack.

    Parameters
    ----------
    stack : list of StaProduct
        Cycle-sorted products from one coregistered stack.
    cycles : sequence of int, optional
        Explicit cycle numbers, e.g. ``(1, 3, 5)``. Overrides all other
        filters.
    max_acquisitions : int, optional
        Hard cap on acquisitions kept.
    max_temporal_baseline : float, optional
        Days. Drop acquisitions beyond this span from the first.
    strategy : {'shortest', 'spread', 'extremes'}, default 'shortest'
        How to thin when ``max_acquisitions`` is below the stack size.

        - ``'shortest'`` — first *n* cycles; minimum temporal decorrelation
        - ``'spread'`` — evenly spaced; maximum :math:`k_z` diversity
        - ``'extremes'`` — first, last, and evenly spaced between

    Returns
    -------
    list of StaProduct
        The selected acquisitions, cycle-sorted.

    Raises
    ------
    ValueError
        If the selection yields fewer than ``MIN_CYCLES`` acquisitions, if
        requested cycles are unavailable, or if ``strategy`` is unknown.

    Notes
    -----
    A minimum of three acquisitions is always enforced. Two give a single
    interferogram — a phase centre, not a vertical profile.
    """
    
    sel = list(stack)

    # explicit cycle selection short-circuits everything else
    if cycles is not None:
        want = set(cycles)
        sel = [p for p in sel if p.cycle in want]
        missing = want - {p.cycle for p in sel}
        if missing:
            raise ValueError(f"Cycles not available: {sorted(missing)}. "
                             f"Stack has C{[p.cycle for p in stack]}")
        if len(sel) < MIN_CYCLES:
            raise ValueError(f"cycles={tuple(cycles)} gives {len(sel)} acquisitions; "
                             f"tomography needs >= {MIN_CYCLES}")
        return sel

    # temporal baseline cap
    if max_temporal_baseline is not None:
        t0 = sel[0].t_start
        keep = [p for p in sel
                if (p.t_start - t0).total_seconds() / 86400 <= max_temporal_baseline + 1e-6]
        if len(keep) < MIN_CYCLES:
            span_needed = (sel[MIN_CYCLES - 1].t_start - t0).total_seconds() / 86400
            raise ValueError(
                f"max_temporal_baseline={max_temporal_baseline} d leaves only "
                f"{len(keep)} acquisition(s). Need >= {span_needed:.0f} d for "
                f"{MIN_CYCLES} acquisitions.")
        log.info("Temporal cap %.0f d: %d -> %d acquisitions",
                 max_temporal_baseline, len(sel), len(keep))
        sel = keep

    # count cap, with thinning strategy
    if max_acquisitions is not None:
        if max_acquisitions < MIN_CYCLES:
            raise ValueError(f"max_acquisitions={max_acquisitions} < {MIN_CYCLES}. "
                             "Two acquisitions give one interferogram, not a profile.")
        if max_acquisitions < len(sel):
            n_keep = max_acquisitions
            if strategy == "shortest":
                idx = list(range(n_keep))
            elif strategy == "spread":
                idx = np.linspace(0, len(sel) - 1, n_keep).round().astype(int).tolist()
            elif strategy == "extremes":
                idx = sorted({0, len(sel) - 1,
                              *np.linspace(0, len(sel) - 1, n_keep).round().astype(int)})[:n_keep]
            else:
                raise ValueError(f"Unknown strategy '{strategy}'")
            kept = [sel[i] for i in sorted(set(idx))]
            log.info("Count cap %d (%s): C%s -> C%s", max_acquisitions, strategy,
                     [p.cycle for p in sel], [p.cycle for p in kept])
            sel = kept

    if len(sel) < MIN_CYCLES:
        raise ValueError(f"Selection yields {len(sel)} acquisitions; need >= {MIN_CYCLES}")
    return sel


# retrieval
def retrieve(products, output_path, auth=None, subdir="l1c", skip_existing=True, max_workers=3):
    """Download and unpack products, returning them with local paths set.

    Parameters
    ----------
    products : list of StaProduct
    output_path : str
        Parent directory. Products land in ``output_path/subdir``.
    auth : MaapAuth, False or None
        ``MaapAuth`` instance to refresh the token; ``False`` to reuse the
        existing ``$MAAP_CATALOGUE_ACCESS_TOKEN``; ``None`` to auto-detect.
    subdir : str, default 'l1c'
    skip_existing : bool, default True
        Skip products already unpacked on disk.
    max_workers : int, default 3
        Concurrent downloads. The zipper endpoint is the bottleneck — keep
        this low.

    Returns
    -------
    list of StaProduct
        Cycle-sorted, each with ``local_dir`` populated.

    Raises
    ------
    RuntimeError
        If ``auth=False`` but no token is present in the environment.
    FileNotFoundError
        If a download reports success but no product directory appears.
    """
    
    # fallback
    download = _get_downloader()

    # authentication
    if auth is False:
        if not os.environ.get("MAAP_CATALOGUE_ACCESS_TOKEN"):
            raise RuntimeError("auth=False but MAAP_CATALOGUE_ACCESS_TOKEN is unset")
        log.info("Using existing MAAP_CATALOGUE_ACCESS_TOKEN")
    elif auth is not None:
        auth.token()
    elif os.environ.get("MAAP_CATALOGUE_ACCESS_TOKEN"):
        log.info("Found MAAP_CATALOGUE_ACCESS_TOKEN — skipping MaapAuth")
    else:
        MaapAuth().token()

    dest = pathlib.Path(output_path) / subdir
    dest.mkdir(parents=True, exist_ok=True)

    todo, out = [], []
    for p in products:
        hits = glob.glob(str(dest / f"{p.item_id}*"))
        if skip_existing and hits:
            log.info("cached C%02d  %s", p.cycle, os.path.basename(hits[0]))
            out.append(StaProduct(**{**p.__dict__, "local_dir": hits[0]}))
        else:
            todo.append(p)

    if todo:
        log.info("Downloading %d product(s) → %s", len(todo), dest)
        kw = dict(disable_bar=False, subdir=subdir)
        if "max_workers" in inspect.signature(download).parameters:
            kw["max_workers"] = max_workers
        download([p.href for p in todo], str(output_path), **kw)

        for p in todo:
            hits = glob.glob(str(dest / f"{p.item_id}*"))
            if not hits:
                hits = glob.glob(str(dest / f"BIO_*C{p.cycle:02d}_T{p.track}_F{p.frame}*"))
            if not hits:
                raise FileNotFoundError(f"Download succeeded but dir missing: {p.item_id}")
            out.append(StaProduct(**{**p.__dict__, "local_dir": hits[0]}))

    return sorted(out, key=lambda x: x.cycle)


# Quadpol Cube
# loading
@dataclass
class BiomassL1CStack:
    """A coregistered, cycle-sorted L1C stack ready for tomography.

    Attributes
    ----------
    products : list of StaProduct
        Cycle-sorted acquisitions sharing one ``stack_key``.
    root : str or None
        Parent directory of the unpacked products.
    """
    
    products: list[StaProduct]
    root: str | None = None
    _shape: tuple | None = field(default=None, repr=False)

    # constructors
    @classmethod
    def from_local(cls, l1c_root: str) -> "BiomassL1CStack":
        """Build a stack from products already present on disk.

        Parameters
        ----------
        l1c_root : str
            Directory containing unpacked ``BIO_*_STA__1S_*`` products.

        Returns
        -------
        BiomassL1CStack

        Raises
        ------
        ValueError
            If fewer than ``MIN_CYCLES`` products are found.
        """
        
        prods = []
        for d in sorted(glob.glob(os.path.join(l1c_root, "BIO_*_STA__1S_*"))):
            m = _STA_RE.search(os.path.basename(d))
            if not m or not os.path.isdir(d):
                continue
            g = m.groupdict()
            prods.append(StaProduct(
                item_id=os.path.basename(d), href="", swath=g["swath"],
                phase=g["phase"], glob=g["glob"], major=g["major"],
                cycle=int(g["cycle"]), track=g["track"], frame=g["frame"],
                baseline=g["baseline"],
                t_start=datetime.strptime(g["start"], "%Y%m%dT%H%M%S"),
                local_dir=d))
        if len(prods) < MIN_CYCLES:
            raise ValueError(f"Only {len(prods)} products in {l1c_root}")
        return cls(products=sorted(prods, key=lambda p: p.cycle), root=l1c_root)

    # properties
    @property
    def n_acq(self) -> int:
        return len(self.products)

    @property
    def cycles(self) -> list[int]:
        return [p.cycle for p in self.products]

    @property
    def dt_days(self) -> np.ndarray:
        """Temporal baselines in days, relative to the first acquisition.

        Returns
        -------
        ndarray of float, shape (n_acq,)
        """
        
        t0 = self.products[0].t_start
        return np.array([(p.t_start - t0).total_seconds() / 86400
                         for p in self.products])

    def validate(self) -> bool:
        """Print a consistency report and confirm stack coherence.

        Checks that all acquisitions share swath, phase, global coverage,
        major cycle, track, frame and baseline; reports cycle contiguity,
        temporal baselines and revisit regularity.

        Returns
        -------
        bool
            True if all fields are consistent and at least ``MIN_CYCLES``
            acquisitions are present.
        """
        
        ok = True
        print(f"--- Stack validation ({self.n_acq} acquisitions) ---")
        for fld in ("swath", "phase", "glob", "major", "track", "frame", "baseline"):
            vals = {getattr(p, fld) for p in self.products}
            good = len(vals) == 1
            ok &= good
            print(f"  {fld:9s}: {sorted(vals)} {'OK' if good else 'MISMATCH'}")
        cyc = self.cycles
        contiguous = cyc == list(range(min(cyc), min(cyc) + len(cyc)))
        print(f"  cycles   : {cyc} {'contiguous' if contiguous else 'GAPS'}")
        dt = self.dt_days
        print(f"  dt [d]   : {np.round(dt).astype(int).tolist()}  (span {dt[-1]:.0f} d)")
        if len(dt) > 1:
            gaps = np.diff(dt)
            print(f"  revisit  : {np.round(gaps).astype(int).tolist()} "
                  f"{'nominal 3-day' if np.allclose(gaps, 3, atol=0.5) else 'irregular'}")
        print(f"  phase    : {self.products[0].phase} "
              f"({'tomographic' if self.products[0].phase == 'T' else 'interferometric'})")
        ok &= self.n_acq >= MIN_CYCLES
        return ok

    # geometry
    def raster_shape(self) -> tuple[int, int]:
        """Return the full-scene raster dimensions.

        Returns
        -------
        tuple of int
            ``(azimuth, range)`` from the first product. Cached after the
            first call.
        """
        
        if self._shape is None:
            gdal.UseExceptions()
            ds = gdal.Open(self._measurement(self.products[0]))
            self._shape = (ds.RasterYSize, ds.RasterXSize)
        return self._shape

    @staticmethod
    def _measurement(p: StaProduct) -> str:
        meas = os.path.join(p.local_dir, "measurement")
        for pat in ("*_i.vrt", "*_i_abs.tiff", "*_abs.tiff"):
            hits = glob.glob(os.path.join(meas, pat))
            if hits:
                return hits[0]
        raise FileNotFoundError(f"No measurement raster in {meas}")

    def lut_path(self, p: StaProduct) -> str:
        """Locate the annotation LUT for one product.

        Parameters
        ----------
        p : StaProduct
            Must have ``local_dir`` set.

        Returns
        -------
        str
            Path to ``annotation_coregistered/*_lut.nc``.

        Raises
        ------
        FileNotFoundError
        """
        
        hits = glob.glob(os.path.join(p.local_dir, "annotation_coregistered", "*_lut.nc"))
        if not hits:
            raise FileNotFoundError(f"No LUT in {p.local_dir}")
        return hits[0]

    def load(self, az_off=0, az_size=None, rg_off=0, rg_size=None,
             pols: Sequence[str] = POLS, mask_sentinel=True,
             dtype=np.complex64) -> tuple[np.ndarray, np.ndarray]:
        """Load the quad-pol cube from local products.

        Parameters
        ----------
        az_off, rg_off : int, default 0
            Window origin in azimuth and range.
        az_size, rg_size : int, optional
            Window extent. Defaults to the remainder of the scene.
        pols : sequence of str, default ('HH', 'HV', 'VH', 'VV')
            Polarisations and their output order.
        mask_sentinel : bool, default True
            Build a validity mask excluding sentinel and non-finite pixels.
        dtype : numpy dtype, default complex64

        Returns
        -------
        S : ndarray, shape (n_acq, n_pol, az, rg)
            Complex SLC cube.
        mask : ndarray of bool, shape (az, rg)
            True where valid in every acquisition and polarisation.

        Notes
        -----
        Polarisation lives in the *band* dimension of the measurement
        raster, not in the filename.
        """

        gdal.UseExceptions()

        full_az, full_rg = self.raster_shape()
        az_size = az_size or full_az - az_off
        rg_size = rg_size or full_rg - rg_off
        win = (rg_off, az_off, rg_size, az_size)

        nbytes = self.n_acq * len(pols) * az_size * rg_size * 8
        log.info("Loading %d acq x %d pol x %d x %d = %.2f GB",
                 self.n_acq, len(pols), az_size, rg_size, nbytes / 1e9)
        if nbytes > 16e9:
            log.warning("Cube > 16 GB — consider a smaller azimuth window")

        S = np.empty((self.n_acq, len(pols), az_size, rg_size), dtype)
        for i, p in enumerate(self.products):
            S[i] = self._read_quadpol(p, win, pols)
            log.info("  C%02d loaded  %s", p.cycle, p.t_start.date())

        if mask_sentinel:
            A = np.abs(S)
            mask = (np.isfinite(A) & (A > 0) & (A < SENTINEL * 0.999)).all(axis=(0, 1))
            log.info("Valid pixels: %.1f%%", mask.mean() * 100)
        else:
            mask = np.ones((az_size, rg_size), bool)
        return S, mask

    @staticmethod
    def _read_quadpol(p: StaProduct, win, pols) -> np.ndarray:
        """Read one acquisition's quad-pol window from disk.

        Prefers the CFloat32 VRT whose bands are named HH/HV/VH/VV; falls
        back to separate ``_abs``/``_phase`` TIFFs recombined as
        :math:`A e^{j\\phi}`.

        Parameters
        ----------
        p : StaProduct
        win : tuple of int
            GDAL window ``(xoff, yoff, xsize, ysize)``.
        pols : sequence of str

        Returns
        -------
        ndarray, shape (n_pol, az, rg), complex64
        """

        meas = os.path.join(p.local_dir, "measurement")

        vrt = glob.glob(os.path.join(meas, "*_i.vrt"))
        if vrt:
            ds = gdal.Open(vrt[0])
            order = {ds.GetRasterBand(b).GetDescription(): b
                     for b in range(1, ds.RasterCount + 1)}
            if all(q in order for q in pols):
                return np.stack([ds.GetRasterBand(order[q]).ReadAsArray(*win)
                                 .astype(np.complex64) for q in pols])

        stem = glob.glob(os.path.join(meas, "*_abs.tiff"))[0][:-len("_abs.tiff")]
        da, dp = gdal.Open(stem + "_abs.tiff"), gdal.Open(stem + "_phase.tiff")
        order = {da.GetRasterBand(b).GetMetadata().get("POLARIMETRIC_INTERP"): b
                 for b in range(1, da.RasterCount + 1)}
        out = []
        for q in pols:
            b = order[q]
            a = da.GetRasterBand(b).ReadAsArray(*win).astype(np.float32)
            ph = dp.GetRasterBand(b).ReadAsArray(*win).astype(np.float32)
            out.append((a * np.exp(1j * ph)).astype(np.complex64))
        return np.stack(out)

    # custom products
    def pauli(self, S: np.ndarray) -> dict[str, np.ndarray]:
        """Compute Pauli channels from a quad-pol cube.

        Parameters
        ----------
        S : ndarray, shape (n_acq, 4, az, rg)

        Returns
        -------
        dict of str to ndarray
            ``'P1'`` surface :math:`(HH+VV)/\\sqrt 2`, ``'P2'`` double-bounce
            :math:`(HH-VV)/\\sqrt 2` locating the **ground**, and ``'XP'``
            volume :math:`(HV+VH)/\\sqrt 2` locating the **canopy**.
        """

        iHH, iHV, iVH, iVV = 0, 1, 2, 3
        return {
            # surface
            "P1": (S[:, iHH] + S[:, iVV]) / np.sqrt(2),
            # double bounce
            "P2": (S[:, iHH] - S[:, iVV]) / np.sqrt(2),
            # volume
            "XP": (S[:, iHV] + S[:, iVH]) / np.sqrt(2),
        }

    def summary(self, S: np.ndarray, mask: np.ndarray) -> None:
        """Print a polarimetric sanity check of the loaded cube.

        Reports per-polarisation median backscatter, dynamic range and
        coefficient of variation, then three physical consistency tests that
        together validate band ordering:

        - ``HV/HH`` should be −14 to −3 dB (canopy depolarisation)
        - ``HV−VH`` should be near 0 dB (reciprocity)
        - ``HH−VV`` should be positive (trunk-ground double bounce)

        Parameters
        ----------
        S : ndarray, shape (n_acq, n_pol, az, rg)
        mask : ndarray of bool, shape (az, rg)
        """
        
        iHH, iHV, iVH, iVV = 0, 1, 2, 3
        names = list(POLS)
        print(f"--- Polarimetric summary (acquisition 0) ---")
        print(f"  cube {S.shape}  valid {mask.mean()*100:.1f}%\n")
        print("  pol | median dB | 5-95% dB |   CV")
        print("  ----+-----------+----------+-------")
        pwr = {}
        for i, p in enumerate(names):
            A = np.abs(S[0, i])[mask]
            db = 20 * np.log10(A)
            p5, p50, p95 = np.percentile(db, [5, 50, 95])
            pwr[p] = 10 * np.log10((A ** 2).mean())
            print(f"  {p}  |  {p50:+7.1f}  |  {p95-p5:6.1f}  | {A.std()/A.mean():.3f}")

        hv_hh = pwr["HV"] - pwr["HH"]
        recip = pwr["HV"] - pwr["VH"]
        hh_vv = pwr["HH"] - pwr["VV"]
        print(f"\n  HV/HH = {hv_hh:+.1f} dB   "
              f"{'OK (canopy depolarisation)' if -14 < hv_hh < -3 else 'CHECK band order'}")
        print(f"  HV-VH = {recip:+.2f} dB   "
              f"{'OK (reciprocity)' if abs(recip) < 1.0 else 'CHECK band order'}")
        print(f"  HH-VV = {hh_vv:+.1f} dB   "
              f"{'OK (trunk-ground double bounce)' if hh_vv > 0 else 'unusual'}")
        print(f"  CV ~ 0.52 expected for single-look amplitude")

    def wavenumbers(self, az_off=0, az_size=None, rg_off=0, rg_size=None,
                    master=None) -> tuple[np.ndarray, np.ndarray, int]:
        """Compute master-referenced vertical wavenumbers from local LUTs.

        Parameters
        ----------
        az_off, rg_off : int, default 0
        az_size, rg_size : int, optional
        master : int, optional
            Index of the reference acquisition. Defaults to the one with
            :math:`k_z` closest to zero.

        Returns
        -------
        KZ : ndarray, shape (n_acq, az, rg), float32
            Vertical wavenumbers in rad/m, with ``KZ[master]`` identically
            zero.
        kz_mean : ndarray, shape (n_acq,)
            Scene-average wavenumber per acquisition.
        master : int
            Index of the reference acquisition.

        Notes
        -----
        Vertical resolution follows from the total spread,
        :math:`\\delta z = 2\\pi/\\Delta k_z`; the height ambiguity follows
        from the smallest gap between adjacent wavenumbers.
        """
        
        full_az, full_rg = self.raster_shape()
        az_size = az_size or full_az - az_off
        rg_size = rg_size or full_rg - rg_off

        KZ = np.stack([
            _interp_lut_to_window(
                _read_lut(self.lut_path(p), "waveNumbers"),
                az_off, az_size, rg_off, rg_size, full_az, full_rg)
            for p in self.products]).astype(np.float32)

        if master is None:
            master = int(np.argmin(np.abs(np.nanmean(KZ, axis=(1, 2)))))
        KZ -= KZ[master]
        kz_mean = np.nanmean(KZ, axis=(1, 2))

        k = np.sort(kz_mean)
        span = k[-1] - k[0]
        dk = np.diff(k)
        dk = dk[dk > 1e-12]
        log.info("master=C%02d  kz span=%.5f rad/m  Rayleigh dz=%.1f m  z_amb=%.1f m",
                 self.products[master].cycle, span,
                 2 * np.pi / span if span > 0 else np.inf,
                 2 * np.pi / dk.min() if dk.size else np.inf)
        return KZ, kz_mean, master

    def to_xarray(self, S, mask=None, az_off=0, rg_off=0, pols=POLS):
        """Wrap a cube as a labelled ``xarray.Dataset``.

        Parameters
        ----------
        S : ndarray, shape (n_acq, n_pol, az, rg)
        mask : ndarray of bool, optional
            Added as a ``(azimuth, range)`` variable.
        az_off, rg_off : int, default 0
            Window origin, used to label the coordinates absolutely.
        pols : sequence of str

        Returns
        -------
        xarray.Dataset
            Dimensions ``(acquisition, azimuth, range)``; variables ``HH``,
            ``HV``, ``VH``, ``VV`` and optionally ``mask``; coordinates
            ``cycle``, ``time`` and ``dt_days`` on the acquisition axis.
        """
    
        n_acq, n_pol, naz, nrg = S.shape
        coords = {
            "acquisition": np.arange(n_acq),
            "azimuth": np.arange(az_off, az_off + naz),
            "range":   np.arange(rg_off, rg_off + nrg),
            "cycle":   ("acquisition", self.cycles),
            "time":    ("acquisition", [p.t_start for p in self.products]),
            "dt_days": ("acquisition", self.dt_days),
        }
        ds = xr.Dataset(
            {p: (("acquisition", "azimuth", "range"), S[:, i])
             for i, p in enumerate(pols)},
            coords=coords,
            attrs=dict(
                track=self.products[0].track, frame=self.products[0].frame,
                swath=self.products[0].swath, phase=self.products[0].phase,
                major_cycle=self.products[0].major,
                baseline=self.products[0].baseline,
                n_acquisitions=n_acq, sentinel=SENTINEL,
                description="BIOMASS L1C coregistered quad-pol stack",
            ),
        )
        if mask is not None:
            ds["mask"] = (("azimuth", "range"), mask)
        for p in pols:
            ds[p].attrs.update(units="complex amplitude",
                               long_name=f"{p} polarisation SLC")
        return ds

    def add_wavenumbers(self, ds, KZ, kz_mean, master):
        """Attach wavenumbers and tomographic geometry to a Dataset.

        Parameters
        ----------
        ds : xarray.Dataset
            Dataset from :meth:`to_xarray`, with dimensions
            ``(acquisition, azimuth, range)``.
        KZ : ndarray, shape (n_acq, az, rg)
            Master-referenced vertical wavenumbers in rad/m.
        kz_mean : ndarray, shape (n_acq,)
            Scene-average wavenumber per acquisition.
        master : int
            Index of the reference acquisition.

        Returns
        -------
        xarray.Dataset
            The input Dataset with a ``kz`` variable, a ``kz_mean``
            coordinate on the acquisition axis, and these attributes:

            ``master_index``, ``master_cycle``
                Position and cycle number of the reference acquisition.
            ``kz_spread``
                :math:`\\max k_z - \\min k_z` in rad/m.
            ``rayleigh_dz``
                Vertical resolution :math:`2\\pi/\\Delta k_z` in metres.
            ``z_ambiguity``
                Unambiguous height range :math:`2\\pi/\\min(\\delta k_z)`
                in metres.

        Notes
        -----
        Non-finite attribute values are replaced by ``0.0``: NetCDF cannot
        serialise infinities.

        Resolution follows from the *total* wavenumber spread, analogous to
        aperture length; ambiguity follows from the *smallest gap* between
        adjacent wavenumbers, analogous to element spacing in an array.
        """
        
        ds["kz"] = (("acquisition", "azimuth", "range"), KZ)
        ds["kz"].attrs.update(units="rad/m",
                              long_name="vertical wavenumber (master-referenced)")
        ds = ds.assign_coords(kz_mean=("acquisition", kz_mean))

        span = float(np.ptp(kz_mean))
        k = np.sort(kz_mean)
        dk = np.diff(k); dk = dk[dk > 1e-12]
        ds.attrs.update(
            master_index=int(master),
            master_cycle=int(self.cycles[master]),
            kz_spread=span,
            rayleigh_dz=float(2*np.pi/span) if span > 0 else 0.0,
            z_ambiguity=float(2*np.pi/dk.min()) if dk.size else 0.0,
        )
        return ds

    def add_pauli(self, ds):
        """Add Pauli decomposition channels to a Dataset.

        Parameters
        ----------
        ds : xarray.Dataset
            Must contain ``HH``, ``HV``, ``VH`` and ``VV``.

        Returns
        -------
        xarray.Dataset
            The input Dataset with three complex variables added:

            ``P1`` : :math:`(HH + VV)/\\sqrt 2`
                Single-bounce / surface scattering.
            ``P2`` : :math:`(HH - VV)/\\sqrt 2`
                Double-bounce from the trunk-ground dihedral, locating the
                **ground** surface.
            ``XP`` : :math:`(HV + VH)/\\sqrt 2`
                Volume scattering from randomly oriented branches, locating
                the **canopy**.

        Notes
        -----
        These are an orthonormal rotation of the scattering vector, so total
        power is conserved. They are *not* written to NetCDF by
        :func:`build_stack` — being exact linear combinations, recomputing
        after :func:`load_dataset` is cheaper than storing them.

        Canopy height follows from the peak separation,
        :math:`h_v = z_{peak}(XP) - z_{peak}(P_2)`, which is insensitive to
        a common phase offset such as a residual flattening screen.
        """
        
        s2 = np.sqrt(2)
        ds["P1"] = (ds.HH + ds.VV) / s2
        ds["P2"] = (ds.HH - ds.VV) / s2
        ds["XP"] = (ds.HV + ds.VH) / s2
        ds["P1"].attrs["long_name"] = "Pauli 1 (HH+VV)/sqrt2 - surface"
        ds["P2"].attrs["long_name"] = "Pauli 2 (HH-VV)/sqrt2 - double bounce, GROUND"
        ds["XP"].attrs["long_name"] = "Cross-pol (HV+VH)/sqrt2 - volume, CANOPY"
        return ds


# LUT helpers
def _read_lut(path: str, var: str) -> np.ndarray:
    """Read one variable from an annotation LUT.

    Tries h5py, xarray and GDAL in turn, returning the first that succeeds.

    Parameters
    ----------
    path : str
        Path to a ``*_lut.nc`` file.
    var : str
        Variable name, e.g. ``'waveNumbers'``.

    Returns
    -------
    ndarray

    Raises
    ------
    RuntimeError
        If no backend can read the variable.

    Notes
    -----
    NetCDF-4 is HDF5 underneath, so h5py works without the ``netCDF4``
    package installed.
    """

    for fn in (_lut_h5py, _lut_xarray, _lut_gdal):
        try:
            a = fn(path, var)
            if a is not None:
                return np.asarray(a)
        except Exception:
            continue
    raise RuntimeError(f"Cannot read '{var}' from {os.path.basename(path)}")


def _lut_h5py(path, var):
    """Read a LUT variable via h5py, searching nested groups.

    Parameters
    ----------
    path : str
    var : str

    Returns
    -------
    ndarray or None
        None if the variable is absent at any depth.
    """

    with h5py.File(path, "r") as f:
        def find(g, name):
            if name in g:
                return np.asarray(g[name][()])
            for k in g:
                if isinstance(g[k], h5py.Group):
                    r = find(g[k], name)
                    if r is not None:
                        return r
            return None
        return find(f, var)


def _lut_xarray(path, var):
    """Read a LUT variable via xarray.

    Parameters
    ----------
    path : str
    var : str

    Returns
    -------
    ndarray
    """
    with xr.open_dataset(path) as ds:
        return ds[var].values


def _lut_gdal(path, var):
    """Read a LUT variable via the GDAL NETCDF driver.

    Parameters
    ----------
    path : str
    var : str

    Returns
    -------
    ndarray
    """

    return gdal.Open(f'NETCDF:"{path}":{var}').ReadAsArray()


def _interp_lut_to_window(a, az_off, az_size, rg_off, rg_size,
                          full_az, full_rg, fill=SENTINEL):
    """Map a coarse full-scene LUT onto a windowed raster subset.

    Parameters
    ----------
    a : ndarray
        Subsampled LUT grid, e.g. 1731x660 for a 20769x1320 scene.
    az_off, rg_off : int
        Window origin in full-scene coordinates.
    az_size, rg_size : int
        Window extent.
    full_az, full_rg : int
        Full-scene raster dimensions the LUT spans.
    fill : float, default 9999.0
        Sentinel marking invalid LUT samples; filled by nearest-neighbour
        before interpolation.

    Returns
    -------
    ndarray, shape (az_size, rg_size)
        LUT values on the window's pixel grid.

    Notes
    -----
    Accounting for ``az_off`` is essential. Interpolating onto the window
    without it stretches the entire scene across the subset and injects a
    spurious ramp.
    """

    a = np.squeeze(np.asarray(a, dtype=np.float64))
    if a.ndim == 3:
        a = a[0]
    if a.ndim == 0:
        return np.full((az_size, rg_size), float(a))

    bad = ~np.isfinite(a) | (a == -fill) | (a == fill)
    if bad.any() and not bad.all():
        idx = distance_transform_edt(bad, return_distances=False, return_indices=True)
        a = a[tuple(idx)]

    f = RegularGridInterpolator(
        (np.linspace(0, full_az - 1, a.shape[0]),
         np.linspace(0, full_rg - 1, a.shape[1])),
        a, bounds_error=False, fill_value=None)
    A, R = np.meshgrid(np.arange(az_off, az_off + az_size),
                       np.arange(rg_off, rg_off + rg_size), indexing="ij")
    return f(np.stack([A.ravel(), R.ravel()], -1)).reshape(az_size, rg_size)


def _download_fallback(urls, output_path, disable_bar=False, subdir="l1c",
                       retries=4, base_delay=5.0, max_workers=3):
    """Download and unzip products in parallel, retrying transient 5xx.

    Used when ``BPS_inputs_download`` is unavailable.

    Parameters
    ----------
    urls : sequence of str
        Product ``.zip`` URLs.
    output_path : str
        Parent directory; archives are unpacked into ``output_path/subdir``
        and then deleted.
    disable_bar : bool, default False
        Suppress progress bars.
    subdir : str, default 'l1c'
    retries : int, default 4
        Attempts per URL before giving up.
    base_delay : float, default 5.0
        Initial backoff in seconds, doubling per attempt.
    max_workers : int, default 3
        Concurrent downloads.

    Raises
    ------
    RuntimeError
        If any product fails all attempts. 4xx responses are not retried.

    Notes
    -----
    The ``/data/zipper/`` endpoint assembles archives on demand and
    intermittently returns 500. The token is re-read from the environment on
    every attempt so a refresh mid-run is picked up.
    """

    pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)
    unzip_path = os.path.join(output_path, subdir)
    urls = list(urls)

    def _one(pos, url):
        fn = url.rsplit('/', 1)[-1] + '.zip'
        out = os.path.join(output_path, fn)
        for attempt in range(retries):
            try:
                # re-read/re-try logic (refreshes token if needed)
                token = os.environ["MAAP_CATALOGUE_ACCESS_TOKEN"]
                r = requests.get(url, headers={"Authorization": f"Bearer {token}"},
                                 stream=True, timeout=(30, 300))
                r.raise_for_status()
                size = int(r.headers.get('content-length', 0))
                with open(out, "wb") as f, tqdm(desc=fn[:40], total=size, unit='iB',
                                                unit_scale=True, unit_divisor=1024,
                                                position=pos, leave=False,
                                                disable=disable_bar) as bar:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        bar.update(f.write(chunk))
                subprocess.run(["unzip", "-q", "-o", out, "-d", unzip_path], check=True)
                os.remove(out)
                log.info("unpacked %s", fn)
                return None
            except requests.HTTPError as e:
                code = e.response.status_code if e.response is not None else None
                if code and code < 500:
                    log.error("%s: HTTP %d — not retryable", fn, code)
                    return url
                if attempt == retries - 1:
                    log.error("%s: HTTP %s after %d attempts", fn, code, retries)
                    return url
                time.sleep(base_delay * (2 ** attempt))
            except Exception as e:
                if os.path.exists(out):
                    os.remove(out)
                if attempt == retries - 1:
                    log.error("%s: %s", fn, type(e).__name__)
                    return url
                time.sleep(base_delay * (2 ** attempt))

    with ThreadPoolExecutor(max_workers=min(max_workers, len(urls))) as ex:
        failed = [f for f in (fut.result() for fut in
                              [ex.submit(_one, i, u) for i, u in enumerate(urls)])
                  if f]

    if failed:
        raise RuntimeError(f"{len(failed)} product(s) failed:\n  " +
                           "\n  ".join(u.rsplit('/', 1)[-1] for u in failed))


def _get_downloader():
    """Return the project downloader, or the built-in fallback.

    Returns
    -------
    callable
        :func:`_download_fallback`.
    """
    return _download_fallback


def aoi_to_tracks(bbox, datetime_range=None, max_items=200,
                  prod_type="S1_STA__1S", catalog_url=CATALOG_URL):
    """Map an AOI to candidate (track, frame) pairs by sampling footprints.

    Parameters
    ----------
    bbox : sequence of float
        ``[W, S, E, N]`` in degrees.
    datetime_range : sequence of str, optional
        ``[start, end]`` ISO-8601. Strongly recommended — an arbitrary
        200-item page may never reach your continent.
    max_items : int, default 200
        Hard cap on items sampled. Higher values raise 502 risk.
    prod_type : str, default 'S1_STA__1S'
    catalog_url : str

    Returns
    -------
    collections.Counter
        ``{(track, frame): n_intersecting}``, most common first.

    Raises
    ------
    RuntimeError
        If no sampled footprint intersects the AOI. The message reports the
        longitude span actually sampled, which usually reveals that the
        window missed the region entirely.

    Notes
    -----
    This is a *sample*, not an exhaustive search: the catalogue 502s under
    sustained pagination. Mid-stream failures are tolerated and the partial
    sample is used. Once resolved, pass ``track=``/``frame=`` directly.
    """
    
    catalog = Client.open(catalog_url)
    kw = dict(collections=L1C_COLLECTIONS, method="GET",
              filter=f"product:type='{prod_type}'", max_items=max_items)
    if datetime_range:
        kw["datetime"] = list(datetime_range)

    search = catalog.search(**kw)
    log.info("Sampling up to %d items of %d matched", max_items, search.matched())

    aoi = box(*bbox)
    hits, lons, n_seen = Counter(), [], 0

    try:
        for it in itertools.islice(search.items(), max_items):
            n_seen += 1
            if it.geometry is None:
                continue
            g = shape(it.geometry)
            lons.append(g.centroid.x)
            if g.intersects(aoi):
                m = _STA_RE.search(it.id)
                if m:
                    hits[(int(m.group("track")), int(m.group("frame")))] += 1
    except Exception as e:
        log.warning("Pagination stopped after %d items (%s) — using partial sample",
                    n_seen, type(e).__name__)

    if not hits:
        lo, hi = (min(lons), max(lons)) if lons else (None, None)
        raise RuntimeError(
            f"No footprints intersect {bbox} in {n_seen} sampled items.\n"
            f"  Sampled longitudes: [{lo}, {hi}]; AOI centroid: {aoi.centroid.x:.1f}\n"
            f"  → the sample likely never reached your region.\n"
            f"  Use track=/frame= directly (Amacayacu = T029 F001).")

    log.info("AOI → %d (track, frame) pair(s) from %d items:", len(hits), n_seen)
    for (t, f), n in hits.most_common(10):
        log.info("  T%03d F%03d: %d", t, f, n)
    return hits


def find_stacks_by_gmtf(track, frame=None, global_id=None, major_cycle=None,
                        prod_type="S1_STA__1S", baseline=None,
                        catalog_url=CATALOG_URL, min_cycles=MIN_CYCLES,
                        return_items=False):
    """CQL2 attribute search. major_cycle and version default to ANY —
    filename-encoded values may be superseded after reprocessing."""
    terms = [f"product:type='{prod_type}'", f"track={int(track)}"]
    if frame is not None:       terms.append(f"frame={int(frame)}")
    if global_id is not None:   terms.append(f"eofeos:global_coverage_id={int(global_id)}")
    if major_cycle is not None: terms.append(f"eofeos:major_cycle_id={int(major_cycle)}")
    if baseline is not None:    terms.append(f"version='{baseline}'")

    catalog = Client.open(catalog_url)
    search = catalog.search(collections=L1C_COLLECTIONS,
                            filter=" and ".join(terms), method="GET")
    log.info("CQL2 matched %d items", search.matched())

    groups, by_id = {}, {}
    for it in dedupe_most_recent(list(search.items())):
        try:
            p = StaProduct.from_item(it)
        except Exception:
            continue
        groups.setdefault(p.stack_key, []).append(p)
        by_id[it.id] = it

    stacks = sorted(
        [sorted({p.cycle: p for p in v}.values(), key=lambda x: x.cycle)
         for v in groups.values() if len(v) >= min_cycles],
        key=lambda s: s[0].t_start, reverse=True)

    for i, s in enumerate(stacks):
        p = s[0]
        log.info("  [%d] M%s v%s | %d cycles C%s | %s → %s (%d d)",
                 i, p.major, p.baseline, len(s), [q.cycle for q in s],
                 p.t_start.date(), s[-1].t_start.date(),
                 (s[-1].t_start - p.t_start).days)

    return (stacks, by_id) if return_items else stacks


def resolve_aoi(bbox, datetime_range=None, max_items=200,
                prod_type="S1_STA__1S", catalog_url=CATALOG_URL,
                return_all=False):
    """Resolve an AOI bounding box to the (track, frame) covering it.

    Parameters
    ----------
    bbox : sequence of float
        ``[W, S, E, N]`` in degrees.
    datetime_range : sequence of str, optional
        Narrows the sample; strongly recommended.
    max_items : int, default 200
        Items sampled before intersecting locally.
    prod_type : str, default 'S1_STA__1S'
    catalog_url : str
    return_all : bool, default False
        Return every matching pair, ranked by hit count.

    Returns
    -------
    tuple of int or list of tuple
        ``(track, frame)``, or a ranked list when ``return_all=True``.

    Notes
    -----
    Server-side bbox filtering returns zero matches on this catalogue, and
    unbounded client-side paging triggers HTTP 502. So a bounded window is
    sampled, footprints are intersected locally, and track/frame are read
    from the product IDs.
    """

    hits = aoi_to_tracks(bbox, datetime_range=datetime_range,
                         max_items=max_items, prod_type=prod_type,
                         catalog_url=catalog_url)
    ranked = [tf for tf, _ in hits.most_common()]
    if return_all:
        return ranked
    t, f = ranked[0]
    if len(ranked) > 1:
        log.info("AOI spans %d (track, frame) pairs; using T%03d F%03d. "
                 "Pass return_all=True to see the rest.", len(ranked), t, f)
    return t, f


# top level functions
def build_stack(output_path=".", *,
                # discovery options, only one out of 4
                reference_id=None,
                track=None, frame=None,
                aoi=None,
                # optional search params
                datetime_range=None, major_cycle=None, global_id=None,
                baseline=None, prod_type="S1_STA__1S",
                # baseline limiting
                cycles=None, max_acquisitions=None,
                max_temporal_baseline=None, max_spatial_baseline=None,
                strategy="shortest",
                # windowing
                az_off=0, az_size=None, rg_off=0, rg_size=None,
                # retrieval mode
                stream=False, save_netcdf=None, download=True, max_workers=4,
                # misc
                stack_index=0, min_cycles=MIN_CYCLES,
                auth=None, verbose=True, aoi_sample=200):
    """Retrieve and pre-build a Biomass L1C tomographic stack.

    Discovery (priority order)
    --------------------------
    reference_id : known product ID — most reliable
    track/frame  : known geometry — reliable  ⭐
    aoi / site   : bbox resolved to track/frame via bounded sample

    Retrieval
    ---------
    stream=False : download products, then read from disk
    stream=True  : windowed HTTPS range reads, no download

    Returns
    -------
    dict: stack, S, mask, KZ, kz_mean, master, dt_days, cycles[, dataset]
    """
    # discover STAC items (only if streaming is enabled)
    items = None

    if reference_id:
        prods, items = find_stack_by_id(reference_id, min_cycles=min_cycles,
                                        return_items=True)

    elif track is not None:
        stacks, items_by_key = find_stacks_by_gmtf(
            track=track, frame=frame, global_id=global_id,
            major_cycle=major_cycle, baseline=baseline,
            prod_type=prod_type, min_cycles=min_cycles, return_items=True)
        if not stacks:
            raise RuntimeError(f"No stack for T{track:03d}"
                               + (f" F{frame:03d}" if frame is not None else ""))
        prods = stacks[stack_index]
        items = [items_by_key[p.item_id] for p in prods]

    elif aoi is not None:
        t, f = resolve_aoi(aoi, datetime_range=datetime_range,
                           max_items=aoi_sample, prod_type=prod_type)
        stacks, items_by_key = find_stacks_by_gmtf(
            track=t, frame=f, global_id=global_id, major_cycle=major_cycle,
            baseline=baseline, prod_type=prod_type,
            min_cycles=min_cycles, return_items=True)
        if not stacks:
            raise RuntimeError(f"AOI -> T{t:03d} F{f:03d} but no stack has "
                               f">= {min_cycles} cycles")
        prods = stacks[stack_index]
        items = [items_by_key[p.item_id] for p in prods]

    else:
        raise ValueError("Specify reference_id=, track= or aoi=")

    # baseline selection
    chosen = select_baseline(prods, cycles=cycles,
                             max_acquisitions=max_acquisitions,
                             max_temporal_baseline=max_temporal_baseline,
                             strategy=strategy)
    if verbose:
        dt = [(p.t_start - chosen[0].t_start).days for p in chosen]
        log.info("Selected %d/%d: C%s | dt %s d", len(chosen), len(prods),
                 [p.cycle for p in chosen], dt)

    # retrieval
    if stream:
        if items is None:
            raise RuntimeError("stream=True requires STAC items — "
                               "use reference_id=, track= or aoi=")
        keep = {p.item_id for p in chosen}
        st = StreamingL1CStack.from_items(
            [it for it in items if it.id in keep], chosen)
        S, mask = st.load_streaming(az_off, az_size, rg_off, rg_size,
                                    max_workers=max_workers)
        KZ, kz_mean, master = st.wavenumbers_streaming(
            az_off, az_size, rg_off, rg_size, max_workers=max_workers)
    else:
        if download:
            chosen = retrieve(chosen, output_path, auth=auth,
                              max_workers=max_workers)
        st = BiomassL1CStack(products=chosen, root=output_path)
        S, mask = st.load(az_off, az_size, rg_off, rg_size)
        KZ, kz_mean, master = st.wavenumbers(az_off, az_size, rg_off, rg_size)

    if verbose and not st.validate():
        log.warning("Stack validation reported mismatches")

    # optional kz filtering
    if max_spatial_baseline is not None:
        keep_i = np.where(np.abs(kz_mean - kz_mean[master]) <= max_spatial_baseline)[0]
        if keep_i.size < MIN_CYCLES:
            raise ValueError(f"max_spatial_baseline leaves {keep_i.size} "
                             f"acquisitions; k_z spread is {np.ptp(kz_mean):.4f} rad/m")
        st.products = [st.products[i] for i in keep_i]
        S, KZ, kz_mean = S[keep_i], KZ[keep_i], kz_mean[keep_i]
        master = int(np.argmin(np.abs(kz_mean)))

    if verbose:
        st.summary(S, mask)
        _report_resolution(kz_mean, st.dt_days)

    result = dict(stack=st, S=S, mask=mask, KZ=KZ, kz_mean=kz_mean,
                  master=master, dt_days=st.dt_days, cycles=st.cycles)

    # saving (optional)
    if save_netcdf:
        ds = st.to_xarray(S, mask, az_off=az_off, rg_off=rg_off)
        ds = st.add_wavenumbers(ds, KZ, kz_mean, master)
        save_dataset(ds, save_netcdf)
        ds = st.add_pauli(ds)
        result["dataset"] = ds
        log.info("saved → %s", save_netcdf)

    return result


def _report_resolution(kz_mean, dt_days):
    """Vertical resolution and ambiguity from the selected baselines."""
    k = np.sort(kz_mean)
    span = k[-1] - k[0]
    dk = np.diff(k); dk = dk[dk > 1e-12]
    print("\n--- Tomographic geometry ---")
    print(f"  acquisitions : {len(kz_mean)}")
    print(f"  k_z spread   : {span:.5f} rad/m")
    print(f"  Rayleigh dz  : {2*np.pi/span:.1f} m" if span > 0 else "  dz: inf")
    if dk.size:
        z_amb = 2*np.pi/dk.min()
        print(f"  z_ambiguity  : {z_amb:.1f} m "
              f"{'OK' if z_amb > 80 else 'WARNING - restrict z grid'}")
    print(f"  temporal span: {dt_days[-1]:.0f} d")
    if len(kz_mean) < 5:
        print(f"  NOTE: {len(kz_mean)} acquisitions is sparse. Capon has few "
              "degrees of freedom; beamformer may perform comparably.")


# Save to NetCDF
def _pick_engine(preferred=None):
    """First available NetCDF engine. Order follows xarray's own fallback [2]."""
    order = ([preferred] if preferred else []) + ["h5netcdf", "netcdf4", "scipy"]
    mods = {"netcdf4": "netCDF4", "h5netcdf": "h5netcdf", "scipy": "scipy"}
    for eng in order:
        if eng is None:
            continue
        try:
            importlib.import_module(mods[eng])
            return eng
        except ImportError:
            continue
    raise RuntimeError("No NetCDF engine. Install one: pip install h5netcdf")


def save_dataset(ds, path, complevel=4, engine=None):
    """Write to NetCDF. Complex variables split into _real/_imag float32 pairs.

    scipy writes NetCDF-3 classic: no compression, no int64, no groups.
    """
    engine = _pick_engine(engine)
    out = ds.copy()

    # complex numbers
    for v in list(out.data_vars):
        if np.iscomplexobj(out[v].values):
            out[f"{v}_real"] = out[v].real.astype("float32")
            out[f"{v}_imag"] = out[v].imag.astype("float32")
            out[f"{v}_real"].attrs["complex_pair"] = f"{v}_imag"
            out = out.drop_vars(v)

    # datetime
    for c in list(out.coords):
        if np.issubdtype(out[c].dtype, np.datetime64):
            out[c].encoding["units"] = "seconds since 2000-01-01"

    if engine == "scipy":
        log.warning("scipy backend -> NetCDF-3: no compression, downcasting int64/bool")
        enc = None
        for v in list(out.data_vars) + list(out.coords):
            if out[v].dtype == np.int64:
                out[v] = out[v].astype(np.int32)
            elif out[v].dtype == bool:
                out[v] = out[v].astype(np.int8)
        # NetCDF-3 attributes
        # replace with something modern in a proper environment
        for tgt in [out] + [out[v] for v in out.variables]:
            for k, val in list(tgt.attrs.items()):
                if val is None or (isinstance(val, float) and not np.isfinite(val)):
                    tgt.attrs[k] = str(val)
    else:
        enc = {v: {"zlib": True, "complevel": complevel} for v in out.data_vars}

    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    log.info("writing %s (engine=%s)", p.name, engine)
    out.to_netcdf(p, engine=engine, encoding=enc)
    log.info("wrote %s (%.1f MB)", p, p.stat().st_size / 1e6)
    return str(p)


def load_dataset(path, engine=None):
    """Reload, recombining _real/_imag pairs into complex."""
    ds = xr.open_dataset(path, engine=_pick_engine(engine))
    for v in [v for v in ds.data_vars if v.endswith("_real")]:
        base = v[:-5]
        ds[base] = ds[v] + 1j * ds[f"{base}_imag"]
        ds = ds.drop_vars([v, f"{base}_imag"])
    return ds


def _token() -> str:
    """Return the current access token from the environment.

    Returns
    -------
    str

    Raises
    ------
    RuntimeError
        If ``$MAAP_CATALOGUE_ACCESS_TOKEN`` is unset.
    """
    
    tok = os.environ.get("MAAP_CATALOGUE_ACCESS_TOKEN")
    if not tok:
        raise RuntimeError("MAAP_CATALOGUE_ACCESS_TOKEN not set — "
                           "run MaapAuth().token() first")
    return tok


ASSET_KEYS = {
    "vrt":   ("enclosure_vrt", "_vrt", ".vrt"),
    "abs":   ("enclosure_i_abs_tiff", "_abs"),
    "phase": ("enclosure_i_phase_tiff", "_phase"),
    "lut":   ("enclosure_nc", "_nc", ".nc"),
    "zip":   ("product",),
}


def _asset_href(item, kind):
    """Resolve a STAC asset href by logical kind.

    Parameters
    ----------
    item : pystac.Item
    kind : {'vrt', 'abs', 'phase', 'lut', 'zip'}
        Logical asset type. Mapped to catalogue keys via ``ASSET_KEYS``.

    Returns
    -------
    str
        Asset URL.

    Raises
    ------
    KeyError
        If no asset matches. The message lists the available keys.

    Notes
    -----
    The catalogue flattens the product directory into ``enclosure_*`` keys,
    so there is no ``lut`` or ``annotation_coregistered`` asset at the STAC
    level — the LUT appears as ``enclosure_nc``.
    """
    
    candidates = ASSET_KEYS.get(kind, (kind,))
    for cand in candidates:
        if cand in item.assets:
            return item.assets[cand].href
    for key, asset in item.assets.items():
        if any(c.strip("._") in key.lower() for c in candidates):
            return asset.href
    raise KeyError(f"No '{kind}' asset; have {sorted(item.assets)}")


def stream_abs_phase(href_abs, href_phase, win, pols=POLS, token=None, retries=4):
    """Read a quad-pol window over HTTPS and recombine amplitude and phase.

    Parameters
    ----------
    href_abs, href_phase : str
        URLs of the amplitude and phase TIFFs.
    win : tuple of int
        ``(rg_off, az_off, rg_size, az_size)``.
    pols : sequence of str, default ('HH', 'HV', 'VH', 'VV')
        Polarisations and their output order.
    token : str, optional
    retries : int, default 4
        Attempts with exponential backoff.

    Returns
    -------
    ndarray, shape (n_pol, az_size, rg_size), complex64
        :math:`S = A e^{j\\phi}`.

    Raises
    ------
    Exception
        Re-raises the last failure after exhausting ``retries``. Transient
        ``Resource temporarily unavailable`` errors from the
        ``/vsipythonfilelike/`` bridge are common and usually clear on retry.

    Notes
    -----
    Band order is read from band descriptions or the
    ``POLARIMETRIC_INTERP`` tag; if absent, positional HH/HV/VH/VV is
    assumed and a warning logged.
    """

    token = token or _token()
    rg_off, az_off, rg_size, az_size = win
    window = Window(col_off=rg_off, row_off=az_off, width=rg_size, height=az_size)

    def _read(href):
        fs = fsspec.filesystem("https",
                               headers={"Authorization": f"Bearer {token}"},
                               block_size=4 * 1024 * 1024)
        with fs.open(href, "rb") as f:
            with rio.open(f) as src:
                order = {}
                for b in range(1, src.count + 1):
                    name = ((src.descriptions[b - 1] or "").upper()
                            or src.tags(b).get("POLARIMETRIC_INTERP", "").upper())
                    if name:
                        order[name] = b
                if not all(p in order for p in pols):
                    order = {p: i + 1 for i, p in enumerate(POLS)}
                    log.warning("Band names absent — assuming HH HV VH VV")
                return np.stack([src.read(order[p], window=window) for p in pols])

    for attempt in range(retries):
        try:
            A = _read(href_abs).astype(np.float32)
            P = _read(href_phase).astype(np.float32)
            return (A * np.exp(1j * P)).astype(np.complex64)
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt
            log.warning("read failed (%s: %s) — retry %d/%d in %ds",
                        type(e).__name__, str(e)[:60], attempt + 1, retries, wait)
            time.sleep(wait)


def fetch_lut(href, cache_dir=".lut_cache", token=None):
    """Download a LUT once and cache it locally.

    Parameters
    ----------
    href : str
        LUT URL.
    cache_dir : str, default '.lut_cache'
    token : str, optional

    Returns
    -------
    str
        Path to the cached file.

    Notes
    -----
    HDF5 range-reading over HTTPS is inefficient — the format requires many
    small seeks through its B-tree index — so the whole file is fetched.
    LUTs are only a few megabytes.

    Writes go to a thread-unique temporary name and are renamed with
    ``os.replace``, which is atomic on POSIX, so concurrent fetches of the
    same LUT cannot corrupt the cache.
    """

    token = token or _token()
    cache = pathlib.Path(cache_dir)
    cache.mkdir(exist_ok=True)
    local = cache / href.rsplit("/", 1)[-1]

    if local.exists():
        return str(local)

    tmp = local.with_suffix(local.suffix + f".{os.getpid()}.{threading.get_ident()}.tmp")
    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"})
    with fs.open(href, "rb") as src, open(tmp, "wb") as dst:
        dst.write(src.read())
    os.replace(tmp, local)                  # atomic on POSIX
    log.info("cached LUT %s (%.1f MB)", local.name, local.stat().st_size / 1e6)
    return str(local)


@dataclass
class StreamingL1CStack(BiomassL1CStack):
    """An L1C stack read directly from the catalogue, without downloading.

    Reads windowed subsets over HTTPS range requests, transferring only the
    pixels requested rather than whole products.

    Attributes
    ----------
    items : list of pystac.Item
        STAC items parallel to ``products``, carrying the asset hrefs.

    See Also
    --------
    BiomassL1CStack : Local equivalent, reading unpacked products from disk.
    """

    items: list = field(default_factory=list, repr=False)

    @classmethod
    def from_items(cls, items, products):
        """Construct from parallel STAC items and parsed products.

        Parameters
        ----------
        items : list of pystac.Item
        products : list of StaProduct
            Must be in the same order as ``items``.

        Returns
        -------
        StreamingL1CStack
        """
        return cls(products=products, items=items)

    def load_streaming(self, az_off=0, az_size=None, rg_off=0, rg_size=None,
                       pols=POLS, mask_sentinel=True, token=None,
                       max_workers=4):
        """Stream the quad-pol cube over HTTPS, acquisitions in parallel.

        Parameters
        ----------
        az_off, rg_off : int, default 0
            Window origin.
        az_size, rg_size : int, optional
            Window extent. Defaults to the remainder of the scene, read from
            the remote header.
        pols : sequence of str, default ('HH', 'HV', 'VH', 'VV')
        mask_sentinel : bool, default True
            Build a validity mask excluding sentinel and non-finite pixels.
        token : str, optional
        max_workers : int, default 4
            Concurrent acquisitions. Each opens two rasters, so the server
            sees ``2 * max_workers`` connections. Keep modest — the
            catalogue returns 502 under load.

        Returns
        -------
        S : ndarray, shape (n_acq, n_pol, az, rg), complex64
        mask : ndarray of bool, shape (az, rg)
            True where valid in every acquisition and polarisation.

        Notes
        -----
        The VRT is skipped deliberately: it references sibling TIFFs by
        relative path, which GDAL cannot resolve through a
        ``/vsipythonfilelike/`` stream. The ``_abs``/``_phase`` pair is read
        instead.

        Each worker writes to its own ``S[i]`` slice, so no locking is
        required. Peak memory exceeds the cube size by roughly
        ``max_workers`` transient per-acquisition buffers.
        """

        token = token or _token()
    
        href0 = _asset_href(self.items[0], "abs")
        full_az, full_rg, n_bands = probe_raster_shape(href0, token)
        log.info("Remote raster: %d az x %d rg, %d bands", full_az, full_rg, n_bands)
    
        az_size = az_size if az_size is not None else full_az - az_off
        rg_size = rg_size if rg_size is not None else full_rg - rg_off
        self._shape = (full_az, full_rg)
        win = (rg_off, az_off, rg_size, az_size)
    
        n = len(self.products)
        log.info("Streaming %d acq x %d pol x %d x %d = %.2f GB (%d workers)",
                 n, len(pols), az_size, rg_size,
                 n * len(pols) * az_size * rg_size * 8 / 1e9, max_workers)
    
        S = np.empty((n, len(pols), az_size, rg_size), np.complex64)
    
        def _one(i):
            item = self.items[i]
            return i, stream_abs_phase(_asset_href(item, "abs"),
                                       _asset_href(item, "phase"),
                                       win, pols, token)
    
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=min(max_workers, n)) as ex:
            futures = {ex.submit(_one, i): i for i in range(n)}
            done = 0
            for fut in as_completed(futures):
                i, arr = fut.result()
                S[i] = arr
                done += 1
                p = self.products[i]
                log.info("  [%d/%d] C%02d streamed  %s", done, n, p.cycle,
                         p.t_start.date())
    
        el = time.time() - t0
        log.info("Streamed %.2f GB in %.1f s (%.1f MB/s)",
                 S.nbytes / 1e9, el, S.nbytes / 1e6 / el)
    
        if mask_sentinel:
            A = np.abs(S)
            mask = (np.isfinite(A) & (A > 0) & (A < SENTINEL * 0.999)).all(axis=(0, 1))
            log.info("Valid pixels: %.1f%%", mask.mean() * 100)
        else:
            mask = np.ones((az_size, rg_size), bool)
        return S, mask

    def wavenumbers_streaming(self, az_off=0, az_size=None, rg_off=0,
                              rg_size=None, master=None, token=None,
                              max_workers=4):
        """Compute master-referenced wavenumbers from remotely fetched LUTs.

        Parameters
        ----------
        az_off, rg_off : int, default 0
        az_size, rg_size : int, optional
        master : int, optional
            Reference acquisition index. Defaults to the one with
            :math:`k_z` closest to zero, which minimises the maximum
            baseline across the stack.
        token : str, optional
        max_workers : int, default 4
            Concurrent LUT fetches.

        Returns
        -------
        KZ : ndarray, shape (n_acq, az, rg), float32
            Wavenumbers in rad/m, with ``KZ[master]`` identically zero.
        kz_mean : ndarray, shape (n_acq,)
        master : int

        Notes
        -----
        Raster dimensions are taken from the cached ``_shape`` if
        :meth:`load_streaming` has already run, otherwise probed from the
        amplitude raster header.
        """

        token = token or _token()
    
        if self._shape is None:
            self._shape = probe_raster_shape(_asset_href(self.items[0], "abs"), token)[:2]
        full_az, full_rg = self._shape
    
        az_size = az_size if az_size is not None else full_az - az_off
        rg_size = rg_size if rg_size is not None else full_rg - rg_off
    
        n = len(self.products)
        KZ = [None] * n
    
        def _one(i):
            local = fetch_lut(_asset_href(self.items[i], "lut"), token=token)
            return i, _interp_lut_to_window(
                _read_lut(local, "waveNumbers"),
                az_off, az_size, rg_off, rg_size, full_az, full_rg)
    
        with ThreadPoolExecutor(max_workers=min(max_workers, n)) as ex:
            for fut in as_completed([ex.submit(_one, i) for i in range(n)]):
                i, arr = fut.result()
                KZ[i] = arr
    
        KZ = np.stack(KZ).astype(np.float32)
    
        if master is None:
            master = int(np.argmin(np.abs(np.nanmean(KZ, axis=(1, 2)))))
        KZ -= KZ[master]
        kz_mean = np.nanmean(KZ, axis=(1, 2))
    
        span = float(np.ptp(kz_mean))
        log.info("master=C%02d  k_z span=%.5f rad/m  Rayleigh dz=%.1f m",
                 self.products[master].cycle, span,
                 2 * np.pi / span if span > 0 else 0.0)
        return KZ, kz_mean, master


def probe_raster_shape(href, token=None):
    """Read raster dimensions from a remote file without fetching pixels.

    Parameters
    ----------
    href : str
        Raster URL.
    token : str, optional
        Bearer token. Defaults to the environment.

    Returns
    -------
    tuple of int
        ``(height, width, count)`` — azimuth lines, range samples, bands.

    Notes
    -----
    Only the header is transferred, a few kilobytes via HTTP range requests.
    """
    
    token = token or _token()
    fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {token}"},
                           block_size=1024 * 1024)
    with fs.open(href, "rb") as f:
        with rio.open(f) as src:
            return src.height, src.width, src.count


In [ ]:
out = build_stack(
    output_path="/data/experiment/my-private-storage/1ec6a172-b859-4d45-8002-eca3aebf3883/",
    track=29, frame=1, major_cycle=5,
    max_acquisitions=7, strategy="spread",
    az_off=8000, az_size=2048, auth=False,
)
S, MASK, KZ = out["S"], out["mask"], out["KZ"]


In [ ]:
out = build_stack(
    track=29, frame=1, major_cycle=5,
    max_acquisitions=7, strategy="spread",
    az_off=8000, az_size=2048,
    stream=True, auth=False,
    max_workers=4,
    save_netcdf="/data/experiment/my-private-storage/1ec6a172-b859-4d45-8002-eca3aebf3883/amacayacu_m05_stack.nc",
)
ds = out["dataset"]


In [ ]:
out["dataset"]